In [ ]:
import io
import os

In [ ]:
import gzip
import pickle #for serializing and de-serializing a Python object structure into text
import numpy as np
import pandas as pd
import urllib.request

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import boto3
import sagemaker #train & deploy models in AWS
import sagemaker.amazon.common as smac #utility to get data in the right format for training
from sagemaker import get_execution_role #access ARN whose credential are used to access ML APIs
from sagemaker.predictor import csv_serializer, json_deserializer #serialize data to our prediction API

# Data Preparation

In [ ]:
with gzip.open('diabetes_data_modified.csv.gz', 'rb') as f:
    train_set, valid_set, test_set = pickle.load(f, encoding='latin1')

In [ ]:
type(train_set)

In [ ]:
print(len(train_set))

In [ ]:
train_set[0].shape #5.000 images, each image is 28x28=784 pixels

In [ ]:
train_set[1][30]

In [ ]:
print(train_set[0].shape)
print(valid_set[0].shape)
print(test_set[0].shape)

In [ ]:
def show_digit(img, caption=''):
    plt.rcParams['figure.figsize'] = (2,10)
    plt.imshow(img.reshape(28,28), cmap='Greys', interpolation='nearest')
    plt.title(caption)
    
show_digit(train_set[0][35], 'This is a {}'.format(train_set[1][35]))

In [ ]:
#Prepare the data to fit in the linear model
vectors = np.array(train_set[0]).astype('float32')
vectors.shape

In [ ]:
labels = np.where(train_set[1] == 3,1,0).astype('float32')
labels.shape

In [ ]:
labels[:20]

END OF DATA PREPARATION
------
BEGIN of training

In [ ]:
buf = io.BytesIO() #the model expected being serialized in a byte form
smac.write_numpy_to_dense_tensor(buf, vectors, labels)



In [ ]:
buf.seek(0) #seek the begining of the buffer

In [ ]:
bucket = 'alexbc-sagemaker-us' #bucket to upload training data
prefix = 'sagemaker/linear-mnist'
key = 'recordio-pb-data'

In [ ]:
resource = boto3.resource('s3')
bucket_obj=resource.Bucket(bucket)
object_res=bucket_obj.Object(os.path.join(prefix, 'train', key)) #reference to the object which hold the data

train_data_loc = 's3://{}/{}/train/{}'.format(bucket, prefix, key)
print('uploaded training data location{}'.format(train_data_loc))

In [ ]:
object_res.upload_fileobj(buf)

In [ ]:
#output of training model dataset
output_location = 's3://{}/{}/output'.format(bucket, prefix)
print('Training artifacts will be uploaded to: {}'.format(output_location))


In [ ]:
region = boto3.Session().region_name
region

In [ ]:
#built-in algorithms are available in containers across different AWS regions
containers = {'us-west-2': '174872318107.dkr.ecr.us-west-2.amazonaws.com/linear-learner:latest',
              'us-east-1': '382416733822.dkr.ecr.us-east-1.amazonaws.com/linear-learner:latest',
              'us-east-2': '404615174143.dkr.ecr.us-east-2.amazonaws.com/linear-learner:latest',
              'eu-west-1': '438346466558.dkr.ecr.eu-west-1.amazonaws.com/linear-learner:latest'}
container = containers[region]

In [ ]:
sess = sagemaker.Session()

In [ ]:
role = get_execution_role()
role

In [ ]:
#Same estimator of tensorflow. Abstracts bias, training, prediction process
linear = sagemaker.estimator.Estimator(container,
                                       role,
                                       train_instance_count=1,
                                       train_instance_type='ml.m5.4xlarge',
                                       output_path=output_location,
                                       sagemaker_session=sess)

In [ ]:
linear.set_hyperparameters(feature_dim=784,
                           predictor_type='binary_classifier',
                           mini_batch_size=200)


In [ ]:
linear.fit({'train': train_data_loc})

--
TRaining completed. It's time for deployment

In [ ]:
linear_predictor = linear.deploy(initial_instance_count=1,
                                 instance_type='ml.m5.large')

In [ ]:
linear_predictor.content_type='text/csv'
linear_predictor.serializer = csv_serializer
linear_predictor.deserializer = json_deserializer

In [ ]:
result = linear_predictor.predict(train_set[0][30])
print(result)

In [ ]:
batched_test_set = np.array_split(test_set[0], 100)

In [ ]:
print(len(test_set[0]))
print(len(batched_test_set))

In [ ]:
batched_test_set[0].shape

In [ ]:
predictions = []
for array in batched_test_set:
    result = linear_predictor.predict(array)
    predictions += [r['predicted_label'] for r in result['predictions']]


In [ ]:
result

In [ ]:
predictions[:50]

In [ ]:
predictions = np.array(predictions)

In [ ]:
# ## Creating the confusion matrix
# * Use the pandas crosstab method to compute a frequency table of two factors
actual = np.where(test_set[1] == 3, 1, 0)

In [ ]:
pd.crosstab(actual, predictions, rownames=['actuals'], colnames=['predictions'])

In [ ]:
sess.delete_endpoint(linear_predictor.endpoint)